# M15 Notebook A: training and federated simulation, full pool, seed 42

Rebuilds the partition and trains every headline method except the LLM ones, then checks each number against the committed results.

**What it runs:**
- the M2 partition;
- E1 and static (M3, M4);
- naïve incremental and local + rehearsal (M6);
- E3, DA-FedDistill (M7);
- M9's K sweep and E4 (`e4_added`);
- E7, federated GBDT (M13).

E2 (FedAvg, FedProx) and FedMD are left out to fit one session; their own notebooks (`m8_kaggle`, `m7_kaggle`) run them.

**The check:** `scripts/m15_check.py --part A` compares every headline cell with the committed seed-42 reports: CSV rows, window average and each window. It passes if every difference is within ±0.005 macro-F1, and writes `reports/m15/full/CHECK_A.md`.

**Attach as input:** the processed CICIoMT2024 dataset (see `release/dataset/README.md`). The first cell checks every file's SHA-256 against `release/dataset/SHA256SUMS`.

**Settings:** Internet *On*, Accelerator *None* (CPU).
**Run:** *Save Version → Save & Run All (Commit)*.

**Time:** about 9.8 h, the sum of measured run times of the same steps in M14. That leaves about 2 h of Kaggle's 12 h session. Every step prints an exit code: 137 means the process was killed, usually for running out of memory.

**Output:** `/kaggle/working/m15_a_seed42_outputs.zip`, rewritten after every step, so it is useful even after a failure. It holds every report file the run changed or added and the run folders.


In [ ]:
SEED = 42   # the check compares against the committed seed-42 reports, so keep 42
import glob, hashlib, os, shlex, shutil, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("scripts/m15_check.py"), "This clone predates M15. Restart the session so it clones the latest main."

# The attached data must be exactly the released processed dataset
assert glob.glob("/kaggle/input/**/data_manifest.json", recursive=True), "No data_manifest.json under /kaggle/input: Add Input -> the processed CICIoMT2024 dataset."
want = {name: sha for sha, name in (line.split() for line in open("release/dataset/SHA256SUMS"))}
for name, sha in want.items():
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} is not under /kaggle/input: attach the processed CICIoMT2024 dataset"
    h = hashlib.sha256()
    with open(hits[0], "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    assert h.hexdigest() == sha, f"{name}: SHA-256 differs from release/dataset/SHA256SUMS, so this is not the released data"
    print("data ok:", name)


In [ ]:
def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True).stdout

def pack():
    """Zip every report file this run changed or added (no checkpoints, no models) and the run folders."""
    stage = "/kaggle/working/m15_stage"
    shutil.rmtree(stage, ignore_errors=True)
    files = sorted(set(git("ls-files", "-m", "--", "reports").split()) | set(git("ls-files", "-o", "--", "reports").split()))
    files = [f for f in files if os.path.isfile(f) and "/checkpoints/" not in f]
    for f in files:
        os.makedirs(os.path.dirname(f"{stage}/{f}"), exist_ok=True)
        shutil.copy2(f, f"{stage}/{f}")
    for r in sorted(glob.glob("/kaggle/working/results/runs/*")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}", ignore=shutil.ignore_patterns("models"))
    os.makedirs(stage, exist_ok=True)
    shutil.make_archive(OUT, "zip", stage)
    shutil.rmtree(stage)
    print(f"zip written: {OUT}.zip ({len(files)} report files)", flush=True)

def step(cmd):
    print("=====", cmd, flush=True)
    rc = subprocess.run(shlex.split(cmd)).returncode
    print("exit code:", rc, flush=True)
    pack()
    return rc


In [ ]:
OUT = f"/kaggle/working/m15_a_seed{SEED}_outputs"
SET = f"--set resources.n_jobs=4 seed={SEED}"   # 4 threads, as in every committed run: results reproduce exactly only at the same thread count
PY = "python -X faulthandler"
ARGS = f"--env kaggle --pool full {SET}"
STEPS = [
    f"{PY} scripts/m2_build_partitions.py --env kaggle --pools full",
    "git checkout -- reports/m2/full/*.png",   # M2 redraws its figures, whose bytes differ on Kaggle; nothing reads them
    f"{PY} scripts/m3_baselines.py {ARGS}",
    f"{PY} scripts/m4_evaluate.py {ARGS}",       # reads the latest M3 run: the one just made
    f"{PY} scripts/m6_incremental.py {ARGS} --resume",
    f"{PY} scripts/m7_distill.py {ARGS} --only e3 --resume",
    f"{PY} scripts/m7_distill.py {ARGS} --report",
    *[f"{PY} scripts/m9_e4.py {ARGS} --only {p} --resume" for p in ("fixed_k10", "fixed_k20", "fixed_k30", "e4_added")],
    f"{PY} scripts/m9_e4.py {ARGS} --report",
    f"{PY} scripts/m13_fedgbdt.py {ARGS}",
    "python scripts/m15_check.py --part A",
]
print("\n".join(STEPS))


In [ ]:
for c in STEPS:
    step(c)


In [ ]:
print(open("reports/m15/full/CHECK_A.md", encoding="utf-8").read() if os.path.exists("reports/m15/full/CHECK_A.md") else "no check report: see the exit codes above")
print("partition files changed by M2 (should be none):", git("status", "--short", "--", "reports/m2/full") or "none")
